# MOSES · round-trip & VUN sanity

Validates the MOSES pipeline: kekulized 4-class round-trip, drop stats, targets, and dataset-row VUN. MOSES counterpart of `01`/`02`.

## Setup

In [ ]:
import os

REPO = "flow-matching-molecules"
if not os.path.isdir(REPO):
    !git clone https://github.com/Nico-Conti/flow-matching-molecules.git
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else ".")

!pip install -q uv
!uv pip install --system -q -e .
import sys; sys.path.insert(0, os.path.abspath("src"))  # flat src/ layout
print("cwd:", os.getcwd())

## 1 · Load MOSES

Build from source (`use_cache=False` forces the sanitize/round-trip pass). `test` split is small for a quick check; switch to `train` for the full ~1.58M.

In [ ]:
import numpy as np
from dataset.moses import load_moses

SPLIT = "test"     # "train" | "test" | "test_scaffolds"
LIMIT = 20000      # None for the full split

moses = load_moses(split=SPLIT, use_cache=False, limit=LIMIT, apply_filter=False)
print("MOSES:", moses["ds"].num_rows, "molecules | split", moses["split"],
      "| bond classes", moses["n_bond_classes"])
print("  atoms:", moses["atom_vocab"], "| targets", moses["targets"])
print("  stats:", moses["stats"])

## 2 · Round-trip (kekulized 4-class)

Neutral encode → strict decode; bonds kekulized (aromatic rings become alternating single/double).

In [ ]:
from rdkit import Chem
from dataset.featurize import smiles_to_tensor, tensor_to_mol, MOSES_ATOMS

def canon(s):
    return Chem.MolToSmiles(Chem.MolFromSmiles(s), isomericSmiles=False)

def check(s):
    X, E = smiles_to_tensor(s, atom_vocab=MOSES_ATOMS, charge_aware=False)
    assert E.shape[-1] == 4 and X.shape[-1] == 7, (X.shape, E.shape)
    m, _ = tensor_to_mol(X, E, atom_vocab=MOSES_ATOMS)
    got = Chem.MolToSmiles(m, isomericSmiles=False) if m is not None else None
    print(f"  {s:26s} -> {str(got):26s} match={got == canon(s)}")

for s in ["c1ccccc1", "c1ccncc1", "O=C(O)c1ccccc1", "c1ccsc1",
          "c1ccc2ccccc2c1", "Clc1ccccc1", "Brc1ccccc1", "CCO"]:
    check(s)

## 3 · Drop summary

In [ ]:
def show(name, stats):
    counts = {k: v for k, v in stats.items() if k.startswith(("drop_", "kept"))}
    if not counts:
        print(f"{name}: {stats}"); return
    n = sum(counts.values())
    kept = counts.get("kept", 0) + counts.get("kept_no_roundtrip", 0)
    print(f"{name}: {kept:,}/{n:,} usable ({kept/n:.2%})")
    for k, v in sorted(counts.items()):
        print(f"    {k:20s} {v:>6,} ({v/n:.2%})")

show("MOSES", moses["stats"])

## 4 · Targets present & normalization stats

In [ ]:
def target_report(name, d):
    y = np.asarray(d["ds"]["y"])
    assert y.shape[0] == d["ds"].num_rows, f"{name}: y rows {y.shape[0]} != #mols {d['ds'].num_rows}"
    n_nan = int(np.isnan(y).sum())
    print(f"{name}: y {y.shape} | targets {d['targets']} | NaNs {n_nan}")
    for j, t in enumerate(d["targets"]):
        c = y[:, j]
        print(f"    {t:6s} mean={c.mean():+.4f}  std={c.std():.4f}  min={c.min():+.4f}  max={c.max():+.4f}")

target_report("MOSES", moses)

## 5 · VUN sanity (dataset rows)

Decoding real MOSES rows should give ~100% validity and high uniqueness (novelty is a self-check here).

In [ ]:
from dataset.metrics import vun_from_graphs

SANITY_N = 5000    # None for the full split
smis = list(moses["ds"]["smiles"])
sample = smis[:SANITY_N] if SANITY_N else smis
graphs = [smiles_to_tensor(s, atom_vocab=MOSES_ATOMS, charge_aware=False)
          for s in sample]
m = vun_from_graphs(graphs, train_smiles=smis, atom_vocab=MOSES_ATOMS, progress=True)
print(f"MOSES: valid={m['validity']:.4f}  unique={m['uniqueness']:.4f}  "
      f"novelty={m['novelty']:.4f} (self-check)  | n={m['n_generated']:,}")

## 6 · Push cleaned split to the Hub *(optional)*

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("HF_TOKEN"), "set HF_TOKEN in .env before pushing"

from dataset.moses import push_moses
push_moses(moses)                    # -> nico8771/moses_<split>
print("pushed moses_" + moses["split"])